In [3]:
import os
import boto3
import time
import pandas as pd
from configparser import ConfigParser
config = ConfigParser(interpolation=None)
config.read('test_config.ini')

['test_config.ini']

In [4]:
username = config['db_details']['username']
password = config['db_details']['password']
hostname = config['db_details']['hostname']
port = config['db_details']['port']
dbname = config['db_details']['dbname']

# Initialize a session using Amazon Athena
session = boto3.Session(
    aws_access_key_id=config['athena_connection']['aws_access_key_id'],
    aws_secret_access_key=config['athena_connection']['aws_secret_access_key'],
    region_name=config['athena_connection']['region_name']
)

database = config['athena_connection']['database']
bucket_name = config['athena_connection']['bucket_name']

usr_database = config['athena_connection']['usr_database']
usr_tbl_name = config['athena_connection']['usr_tbl_name']

In [11]:
query = "SELECT * FROM brand_choice_count;"
#query = "SELECT DISTINCT therapeutic_area FROM cp_ldscp_sum_intell_prop;"

In [18]:
from typing import Dict, List, Tuple

def extract_schema(database_name: str):
    glue = session.client('glue')
    try:
        tables = glue.get_tables(DatabaseName=database_name)['TableList']
    except Exception as e:
        print(f"Error fetching tables from Glue: {e}")
        return {}
    
    schema = {}

    for table in tables:
        table_name = table.get('Name')
        if not table_name:
            print("Table missing 'Name' key")
            continue

        try:
            storage_desc = table.get('StorageDescriptor', {})
            raw_columns = storage_desc.get('Columns', [])
            raw_partition_keys = table.get('PartitionKeys', [])

            # Process regular columns
            columns = {}
            for col in raw_columns:
                col_name = col.get('Name')
                if not col_name:
                    print("Column missing 'Name' key")
                    continue

                col_type = col.get('Type', 'unknown')
                col_desc = col.get('Comment', 'No description')
                columns[col_name] = {
                    'Type': col_type,
                    'Description': col_desc,
                    'IsPartitionKey': False
                }

            # Process partition keys
            for pkey in raw_partition_keys:
                pkey_name = pkey.get('Name')
                if not pkey_name:
                    print("Partition key missing 'Name' key")
                    continue

                pkey_type = pkey.get('Type', 'unknown')
                pkey_desc = pkey.get('Comment', 'No description')
                columns[pkey_name] = {
                    'Type': pkey_type,
                    'Description': pkey_desc,
                    'IsPartitionKey': True
                }

            description = table.get('Description', 'No description')
            location = storage_desc.get('Location', '')

            schema[table_name] = {
                'columns': columns,
                'description': description,
                'location': location
            }

        except Exception as e:
            print(f"Error processing table {table_name}: {e}")

    return schema

def get_schema_dtls(schema_data: Dict[str, Dict]) -> Tuple[List[str], List[Dict]]:
        """
        Prepare schema chunks and metadata for embedding.
        """
        schema_chunks = []
        metadata_map = []

        for table_name, table_info in schema_data.items():
            # Include column names in table-level chunk
            column_names = ", ".join(table_info['columns'].keys())
            table_chunk = f"Table: {table_name}. Description: {table_info['description']}. Columns: {column_names}"
            
            schema_chunks.append(table_chunk)
            metadata_map.append({
                'type': 'table',
                'name': table_name,
                'columns': list(table_info['columns'].keys())
            })

        return schema_chunks, metadata_map

In [17]:
schema_info = extract_schema(database_name="precision_growth")
schema_info.keys()


dict_keys(['brand_choice_count', 'cp_ldscp_sum_intell_prop', 'cp_ldscp_sum_mkt_val', 'cp_ldscp_sum_mkt_val_top_records', 'cp_ldscp_sum_prod_vs_class', 'cp_ldscp_sum_prod_vs_class_top_records', 'cp_ldscp_sum_prod_vs_ind', 'drug_mapping_buying_process', 'indication_demographic_lkp_biz', 'mkt_map_acc_copay_ins_class', 'mkt_map_acc_copay_ins_growth_class', 'mkt_map_acc_copay_ins_growth_indication', 'mkt_map_acc_copay_ins_indication', 'mkt_map_acc_pat_ins_type', 'mkt_map_acc_pat_ins_type_growth_class', 'mkt_map_acc_pat_ins_type_pit_class', 'mkt_map_acc_pat_ins_type_pit_indication', 'mkt_map_acc_reject_rate_growth_class', 'mkt_map_acc_reject_rate_growth_ind', 'mkt_map_acc_reject_rate_pit_class', 'mkt_map_acc_reject_rate_pit_ind', 'mkt_map_acc_reject_reason', 'mkt_map_adh_persistency_curve', 'mkt_map_brand_choice', 'mkt_map_brand_choice_new', 'mkt_map_brand_to_adherence_count', 'mkt_map_brand_to_adherence_percentage', 'mkt_map_brand_to_adherence_scatterplot', 'mkt_map_buy_proc_class_brd_adh',

In [19]:
schema_chunks, metadata = get_schema_dtls(schema_info)
metadata

[{'type': 'table',
  'name': 'brand_choice_count',
  'columns': ['indication',
   'regimen',
   'line_number',
   'distinct_count_of_patients',
   'time_period',
   'therapeutic_area']},
 {'type': 'table',
  'name': 'cp_ldscp_sum_intell_prop',
  'columns': ['product_name',
   'indication_name',
   'company',
   'pharmacological_class',
   'moa',
   'roa',
   'phase',
   'therapeutic_area']},
 {'type': 'table',
  'name': 'cp_ldscp_sum_mkt_val',
  'columns': ['product_name',
   'indication',
   'company',
   'pharmacological_class',
   'launch_year',
   'loe_year',
   'product_revenue_year',
   'product_revenue_annual',
   'product_revenue_last_full_cy',
   'product_revenue_cy_plus_max',
   'product_cagr',
   'percent_market_share_cy_plus_max',
   'therapeutic_area']},
 {'type': 'table',
  'name': 'cp_ldscp_sum_mkt_val_top_records',
  'columns': ['product_name',
   'indication',
   'product_revenue_year',
   'top_5_product',
   'product_revenue_last_full_cy',
   'product_revenue_annual',

In [12]:
athena_client = session.client('athena')
wait_time = 2

response = athena_client.start_query_execution(
    QueryString=query,
    QueryExecutionContext={'Database': database},
    ResultConfiguration={'OutputLocation': f's3://{bucket_name}/athena-results/'}
)
query_execution_id = response['QueryExecutionId']


query_status = athena_client.get_query_execution(QueryExecutionId=query_execution_id)['QueryExecution']['Status']['State']
max_retries = 3
try:
    df = None
    athena_client = session.client('athena')

    response = athena_client.start_query_execution(
        QueryString=query,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={'OutputLocation': f's3://{bucket_name}/athena-results/'}
    )
    query_execution_id = response['QueryExecutionId']

    for attempt in range(max_retries):
        query_status = athena_client.get_query_execution(QueryExecutionId=query_execution_id)['QueryExecution']['Status']['State']
        if query_status == 'SUCCEEDED':
            results = athena_client.get_query_results(QueryExecutionId=query_execution_id)
            columns = [col['Label'] for col in results['ResultSet']['ResultSetMetadata']['ColumnInfo']]
            data_rows = [
                [field.get('VarCharValue', None) for field in row['Data']]
                for row in results['ResultSet']['Rows'][1:]
            ]
            df =  pd.DataFrame(data_rows, columns=columns)
        elif query_status in ['FAILED', 'CANCELLED']:
            print(f"Query failed: {query_status}")
            
        time.sleep(wait_time)
        wait_time = min(wait_time * 2, 10)  # Exponential backoff
except Exception as e:
    print(f"Error executing Athena query: {e}")

In [13]:
df

,indication,regimen,line_number,distinct_count_of_patients,time_period,therapeutic_area
0,Eczema/Dermatitis,Corticosteroid + Vtama + Winlevi,3,20,None,eczema_dermatitis
1,Eczema/Dermatitis,Fasenra,3,20,None,eczema_dermatitis
2,Eczema/Dermatitis,Calcineurin inhibitor + Corticosteroid + Olumiant,2,20,None,eczema_dermatitis
3,Eczema/Dermatitis,Glucocorticoid receptor (GCR) agonist + Skyrizi,2,20,None,eczema_dermatitis
4,Eczema/Dermatitis,Corticosteroid + Opzelura + Rinvoq,2,20,None,eczema_dermatitis
...,...,...,...,...,...,...
994,Multiple myeloma,Darzalex + Kyprolis,1,12,None,multiple_myeloma
995,Multiple myeloma,Prolia + Venclexta,4,12,None,multiple_myeloma
996,Multiple myeloma,Prolia + Proteasome inhibitor,7,8,None,multiple_myeloma
997,Multiple myeloma,Darzalex + Prolia,4,12,None,multiple_myeloma
